# 20. NM Assessment Verification

Verify the NM assessment with the default cross section from scite_app.py:
- 300x500mm rectangular section
- C30/37 concrete with EC2ParabolaRectangle (design values)
- 4xD20 B500B bars at z=50mm (design values)

This notebook investigates the steel material model representation issue.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scite.matmod import EC2ParabolaRectangle, create_steel
from scite.cs_design import CrossSection, ReinforcementLayer, RectangularShape, ReinforcementLayout
from scite.nm_assess import NMAssessment

print("Imports successful")

## Step 1: Check Steel Material Model

First, let's create a steel material and check its attributes.

In [ ]:
# Create steel material using the factory function
steel = create_steel('B500B', gamma_s=1.15)

print("Steel material created successfully")
print(f"\nType: {type(steel)}")
print(f"\nAvailable attributes:")
for attr in dir(steel):
    if not attr.startswith('_'):
        print(f"  - {attr}")

print(f"\n=== Design Values ===")
print(f"f_yk = {steel.f_yk:.1f} MPa (characteristic yield)")
print(f"f_yd = {steel.f_yd:.1f} MPa (design yield)")
print(f"f_tk = {steel.f_tk:.1f} MPa (characteristic tensile)")
print(f"f_td = {steel.f_td:.1f} MPa (design tensile)")
print(f"γ_s = {steel.gamma_s:.2f}")
print(f"eps_yd = {steel.eps_yd:.6f} (design yield strain)")
print(f"eps_ud = {steel.eps_ud:.4f} (ultimate strain)")

# Check if old attributes exist
print(f"\n=== Checking old attributes ===")
print(f"Has 'f_sy': {hasattr(steel, 'f_sy')}")
print(f"Has 'f_st': {hasattr(steel, 'f_st')}")
print(f"Has 'eps_sy': {hasattr(steel, 'eps_sy')}")

## Step 2: Create Default Cross Section

Create the same cross section as in scite_app.py

In [ ]:
# Create concrete material - C30/37 with design values
concrete = EC2ParabolaRectangle(f_ck=30, alpha_cc=1.0, gamma_c=1.5)

print("Concrete material:")
print(f"  f_ck = {concrete.f_ck:.1f} MPa")
print(f"  f_cd = {concrete.f_cd:.1f} MPa")
print(f"  α_cc = {concrete.alpha_cc}")
print(f"  γ_c = {concrete.gamma_c}")

# Create steel material - B500B with design values
steel = create_steel('B500B', gamma_s=1.15)

print("\nSteel material:")
print(f"  f_yk = {steel.f_yk:.1f} MPa")
print(f"  f_yd = {steel.f_yd:.1f} MPa")
print(f"  γ_s = {steel.gamma_s}")

In [ ]:
# Create shape, reinforcement layout, and cross section
shape = RectangularShape(b=300.0, h=500.0)
reinforcement = ReinforcementLayout()

cs = CrossSection(
    shape=shape,
    concrete=concrete,
    reinforcement=reinforcement
)

print(f"Cross section created: {cs.shape.b}x{cs.shape.h} mm")
print(f"Height: {cs.h_total:.1f} mm")

In [ ]:
# Add reinforcement layer - 4xD20 at c_nom=50mm from bottom
# For h=500mm, bottom reinforcement at 50mm from bottom means z=50mm from bottom
# (Note: verify coordinate system - z might be from top or bottom depending on implementation)
c_nom = 50.0  # Cover [mm]
n_bars = 4
d_bar = 20.0  # D20 bars [mm]
A_s = n_bars * np.pi * (d_bar/2)**2  # Total steel area

layer = ReinforcementLayer(
    name='Bottom reinforcement',
    z=c_nom,  # Same as scite_app.py: z=50mm
    A_s=A_s,
    material=steel
)

cs.reinforcement.layers.append(layer)

print(f"\nReinforcement layer added:")
print(f"  Name: {layer.name}")
print(f"  z = {layer.z:.1f} mm")
print(f"  Cover c_nom = {c_nom:.1f} mm")
print(f"  Bars: {n_bars} × D{d_bar:.0f}")
print(f"  A_s = {layer.A_s:.1f} mm²")
print(f"  Material type: {type(layer.material).__name__}")

# Check layer properties that might reference f_sy
print(f"\nLayer properties:")
try:
    print(f"  f_sy (from layer): {layer.f_sy:.1f} MPa")
except AttributeError as e:
    print(f"  ERROR accessing f_sy: {e}")

try:
    print(f"  f_st (from layer): {layer.f_st:.1f} MPa")
except AttributeError as e:
    print(f"  ERROR accessing f_st: {e}")

try:
    print(f"  eps_sy (from layer): {layer.eps_sy:.6f}")
except AttributeError as e:
    print(f"  ERROR accessing eps_sy: {e}")

## Step 3: Visualize Cross Section

In [ ]:
# Plot cross section geometry
fig, ax = plt.subplots(figsize=(6, 8))

# Plot shape
patches = cs.shape.get_plot_patches()
for patch, label in patches:
    ax.add_patch(patch)

# Plot reinforcement layers
for layer in cs.reinforcement.layers:
    # Plot as horizontal line at z coordinate (distance from top)
    y_coord = layer.z  # z is measured from top
    ax.plot(0, y_coord, 'ro', markersize=10, label=f'{layer.name} ({layer.A_s:.0f} mm²)')

ax.set_xlim(cs.shape.get_plot_xlim())
ax.set_ylim(cs.shape.get_plot_ylim())
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_xlabel('Width [mm]')
ax.set_ylabel('Height [mm]')
ax.set_title('Cross Section: 300x500mm with 4xD20')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nCross section info:")
print(f"  Number of layers: {len(cs.reinforcement.layers)}")
print(f"  Total steel area: {cs.A_s:.1f} mm²")

## Step 4: Create NM Assessment

In [ ]:
# Create NM assessment with external loads
try:
    nm = NMAssessment(
        cs=cs,
        N_Ed=-50.0,  # 500 kN compression
        M_Ed=100.0,   # 100 kNm bending
        eps_top=-0.0035,  # Initial guess
        eps_bot=0.0078
    )
    
    print("NM Assessment created successfully!")
    print(f"\nExternal loads:")
    print(f"  N_Ed = {nm.N_Ed:.1f} kN")
    print(f"  M_Ed = {nm.M_Ed:.1f} kNm")
    
    print(f"\nStrain state:")
    print(f"  ε_top = {nm.eps_top:.6f}")
    print(f"  ε_bot = {nm.eps_bot:.6f}")
    
    print(f"\nInternal forces:")
    print(f"  N_actual = {nm.N_actual:.1f} kN")
    print(f"  M_actual = {nm.M_actual:.1f} kNm")
    
    print(f"\nEquilibrium errors:")
    print(f"  N_error = {nm.N_error:.1f} kN")
    print(f"  M_error = {nm.M_error:.1f} kNm")
    
except Exception as e:
    print(f"ERROR creating NMAssessment: {e}")
    import traceback
    traceback.print_exc()

## Step 5: Visualize Stress-Strain Profile

In [ ]:
# Plot stress-strain profile
try:
    fig, (ax_strain, ax_stress) = plt.subplots(1, 2, figsize=(14, 8), sharey=True)
    
    nm.profile.plot_stress_strain_profile(
        ax_strain=ax_strain,
        ax_stress=ax_stress,
        show_resultants=True,
        concrete_label='F_cd',
        steel_label='F_sd',
        N_Ed=nm.N_Ed,
        M_Ed=nm.M_Ed,
        show_assessment=True
    )
    
    plt.tight_layout()
    plt.show()
    
    print("Stress-strain profile plotted successfully")
    
except Exception as e:
    print(f"ERROR plotting profile: {e}")
    import traceback
    traceback.print_exc()

## Step 6: Search for f_sy References

Let's check where f_sy might still be referenced in the codebase.

In [ ]:
# Check reinforcement layer summary that might use f_sy
print("Checking layer string representation and summary methods:")
print(f"\nLayer repr: {repr(layer)}")
print(f"\nLayer str: {str(layer)}")

# Try to get layer summary if it exists
if hasattr(layer, 'summary'):
    try:
        summary = layer.summary()
        print(f"\nLayer summary: {summary}")
    except Exception as e:
        print(f"ERROR in layer.summary(): {e}")
        import traceback
        traceback.print_exc()

# Check cross section summary
if hasattr(cs, 'summary'):
    try:
        cs_summary = cs.summary()
        print(f"\nCross section summary: {cs_summary}")
    except Exception as e:
        print(f"ERROR in cs.summary(): {e}")
        import traceback
        traceback.print_exc()

## Summary

This notebook verifies:
1. Steel material model uses new design-based attributes (f_yk, f_yd, eps_yd)
2. Cross section creation with design values for both concrete and steel
3. NM assessment with external loads N_Ed and M_Ed
4. Equilibrium error calculation (N_err = N_internal - N_Ed)
5. Any remaining references to old f_sy attribute